Step 1: match the schools

In [1]:
from pathlib import Path
import pandas as pd, re, difflib

here = Path.cwd()
repo_root = next(p for p in [here, *here.parents] if (p / "data").exists())

co = pd.read_csv(repo_root / "data" / "colorado_data" / "year_1_clean.csv", dtype={"ENTITY_ID": str})
feat = pd.read_csv(repo_root / "data" / "victoria_datasets" / "clean_dataset.csv",
                   dtype={"COUNTY_FIPS": str}, low_memory=False)

schools_co = co[["ENTITY_ID", "ENTITY_NAME"]].drop_duplicates()
cands = (feat.loc[feat["STABBR"] == "CO", ["UNITID", "INSTNM_inst"]]
             .dropna().drop_duplicates())
cands["UNITID"] = cands["UNITID"].astype("Int64")
print(len(schools_co), "Colorado schools in the earnings file;",
      len(cands), "Colorado schools in our Scorecard data")

def norm(s):
    s = s.lower().replace("&", "and")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\b(the|of|at)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()

cands["key"] = cands["INSTNM_inst"].map(norm)
rows = []
for _, r in schools_co.iterrows():
    k = norm(r["ENTITY_NAME"])
    best = difflib.get_close_matches(k, list(cands["key"]), n=1, cutoff=0.0)[0]
    m = cands[cands["key"] == best].iloc[0]
    rows.append({"ENTITY_ID": r["ENTITY_ID"], "colorado_name": r["ENTITY_NAME"],
                 "scorecard_name": m["INSTNM_inst"], "UNITID": m["UNITID"],
                 "score": round(difflib.SequenceMatcher(None, k, best).ratio(), 2)})
match = pd.DataFrame(rows).sort_values("score")
match

27 Colorado schools in the earnings file; 64 Colorado schools in our Scorecard data


,ENTITY_ID,colorado_name,scorecard_name,UNITID,score
21,4504,CSU-Global Campus,Cheeks Beauty Academy,381699,0.42
22,0534,University of Denver,Metropolitan State University of Denver,127565,0.64
0,0511,Community College of Denver,Community College of Denver,126942,1.00
24,0524,Colorado State University - Pueblo,Colorado State University Pueblo,128106,1.00
23,0536,Western Colorado University,Western Colorado University,128391,1.00
20,0544,Morgan Community College,Morgan Community College,127617,1.00
19,5124,Emily Griffith Technical College,Emily Griffith Technical College,127158,1.00
18,0519,Metropolitan State University of Denver,Metropolitan State University of Denver,127565,1.00
17,0523,Colorado Christian University,Colorado Christian University,126669,1.00
16,0505,Aims Community College,Aims Community College,126207,1.00


In [2]:
print(cands[cands["INSTNM_inst"].str.contains("Global|Denver|Colorado State", case=False)][["UNITID", "INSTNM_inst"]])
print()
print(match[match.duplicated("UNITID", keep=False)])

       UNITID                                        INSTNM_inst
6492   126562  University of Colorado Denver/Anschutz Medical...
6610   126942                        Community College of Denver
6634   126951               Lincoln College of Technology-Denver
6751   127565            Metropolitan State University of Denver
6940   128106                   Colorado State University Pueblo
36390  404912                      Pima Medical Institute-Denver
39789  459277                             Aveda Institute-Denver
40386  476975                   Colorado State University Global
40674  482176                    Paul Mitchell the School-Denver

   ENTITY_ID                            colorado_name  \
22      0534                     University of Denver   
18      0519  Metropolitan State University of Denver   

                             scorecard_name  UNITID  score  
22  Metropolitan State University of Denver  127565   0.64  
18  Metropolitan State University of Denver  127565   1

Step 2: fix the matches and join the Colorado earnings to our features

In [3]:
# 1. Fix the two bad matches
match.loc[match["colorado_name"] == "CSU-Global Campus", ["scorecard_name", "UNITID"]] = ["Colorado State University Global", 476975]
match = match[match["colorado_name"] != "University of Denver"].copy()
match["UNITID"] = match["UNITID"].astype("Int64")
print(len(match), "schools matched; UNITIDs unique:", match["UNITID"].is_unique)

# 2. Attach UNITID to the Colorado programs
co["CIPCODE"] = co["CIPCODE"].astype("Int64")
co2 = co.merge(match[["ENTITY_ID", "UNITID"]], on="ENTITY_ID", how="inner")
print("Colorado programs:", len(co), "-> at matched schools:", len(co2))

# 3. Join to our features on school + 4-digit CIP code
feat["UNITID"] = feat["UNITID"].astype("Int64")
feat["CIPCODE"] = feat["CIPCODE"].astype("Int64")
print("duplicate school+CIP rows in our features:", feat.duplicated(["UNITID", "CIPCODE"]).sum())

joined = co2.merge(feat, on=["UNITID", "CIPCODE"], how="left", indicator=True)
print(joined["_merge"].value_counts().to_string())

has_e = joined["EARN_MEDIAN"].notna()
ok = has_e & (joined["_merge"] == "both")
print("programs with earnings:", has_e.sum())
print("...and also found in our features:", ok.sum())
print("schools in that usable set:", joined.loc[ok, "UNITID"].nunique())

26 schools matched; UNITIDs unique: True
Colorado programs: 747 -> at matched schools: 743
duplicate school+CIP rows in our features: 805
_merge
both          623
left_only     120
right_only      0
programs with earnings: 347
...and also found in our features: 343
schools in that usable set: 22


Step 3: build the label and check it before training

In [4]:
d = joined[ok].copy()

# Label: did Year 1 median earnings beat the county's high school baseline?
d["BASELINE"] = d["B20004_003E"]
print("programs with no baseline:", d["BASELINE"].isna().sum())
d = d.dropna(subset=["BASELINE"])
d["LABEL"] = (d["EARN_MEDIAN"] > d["BASELINE"]).astype(int)

print("\nprograms:", len(d), "| share labeled 1:", round(d["LABEL"].mean(), 2))
print("\nearnings minus baseline ($):")
print((d["EARN_MEDIAN"] - d["BASELINE"]).describe().round(0).to_string())

print("\nby certificate length:")
print(d.groupby("DEGREE")["LABEL"].agg(["size", "mean"]).round(2).to_string())

print("\nby school:")
print(d.groupby("ENTITY_NAME")["LABEL"].agg(["size", "mean"]).sort_values("size", ascending=False).round(2).to_string())

# Sanity check: do Colorado earnings agree with Scorecard earnings for the same programs?
both = d[["EARN_MEDIAN", "EARN_MDN_1YR_CLEAN"]].dropna()
print("\nprograms with both Colorado and Scorecard earnings:", len(both))
if len(both) > 5:
    print("correlation:", round(both["EARN_MEDIAN"].corr(both["EARN_MDN_1YR_CLEAN"], method="spearman"), 2))

programs with no baseline: 0

programs: 343 | share labeled 1: 0.42

earnings minus baseline ($):
count      343.0
mean       692.0
std      13232.0
min     -21598.0
25%      -8618.0
50%      -2749.0
75%       7890.0
max      73831.0

by certificate length:
                      size  mean
DEGREE                          
Certificate(<1 year)   241  0.36
Certificate(>1 year)   102  0.55

by school:
                                         size  mean
ENTITY_NAME                                        
Front Range Community College              39  0.38
Red Rocks Community College                32  0.44
Pikes Peak State College                   32  0.34
Emily Griffith Technical College           28  0.50
Arapahoe Community College                 26  0.42
Pueblo Community College                   21  0.48
Aims Community College                     20  0.35
Colorado Mountain College                  18  0.50
Community College of Aurora                17  0.47
Community College of Denve

Step 4: save the table and create the Colorado setup file

In [5]:
keep = ["UNITID", "CIPCODE", "ENTITY_NAME", "DEGREE", "EARN_MEDIAN", "BASELINE", "LABEL"]
feat_cols = [c for c in d.columns if c in feat.columns and c not in ("UNITID", "CIPCODE")]
out = d[keep + feat_cols].copy()
out.to_csv(repo_root / "data" / "colorado_data" / "colorado_model_table.csv", index=False)
print(out.shape)

(343, 40)
